# Register agents in entitycore

Agents (authors + affiliations) for: Shiu, Sterne et al. (2024), *A Drosophila computational brain model reveals sensorimotor processing*, **Nature**. DOI: [10.1038/s41586-024-07763-9](https://www.nature.com/articles/s41586-024-07763-9).

Reads from the local spreadsheet `Drosophila_brain_model_agents.xlsx` (sheets: `persons`, `organizations`, `consortiums`).

In [ ]:
import pandas as pd
import os

from datetime import datetime
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token

In [ ]:
token = get_token(environment="staging")
project_context = ProjectContext.from_vlab_url("https://staging.openbraininstitute.org/app/virtual-lab/lab/1f91f30e-1489-4e2a-8eb7-1217257c8e19/project/7a411785-6895-4839-aaa2-d9f76e09875a/home")
client = Client(environment="staging", project_context=project_context, token_manager=token)

# token = get_token(environment="production")
# project_context = ProjectContext.from_vlab_url("https://www.openbraininstitute.org/app/virtual-lab/lab/5f8376bf-b84f-4188-8ef5-e1df3d7529b4/project/7d22829c-edc6-4b1d-8ab9-99dd9e511e74/home")
# client = Client(environment="production", project_context=project_context, token_manager=token)

In [ ]:
# Spreadsheet with the agents (persons / organizations / consortiums) to be registered.
# Generated from the author list of DOI 10.1038/s41586-024-07763-9.
agents_file = "Drosophila_brain_model_agents.xlsx"  # relative to this notebook's folder
output_root = "."  # where the registered_*.csv outputs are written

### Load list of agents to be registered

In [ ]:
persons = pd.read_excel(agents_file, sheet_name="persons")
organizations = pd.read_excel(agents_file, sheet_name="organizations")
consortiums = pd.read_excel(agents_file, sheet_name="consortiums")
persons["ID"] = "-"
organizations["ID"] = "-"
consortiums["ID"] = "-"

print(f"#Persons: {persons.shape[0]}")
print(f"#Organizations: {organizations.shape[0]}")
print(f"#Consortiums: {consortiums.shape[0]}")

### Register agents

In [ ]:
check_only = True  # Set to True to check without actually registering

#### Persons

In [ ]:
for i, person in persons.iterrows():
    # Skip if already registered
    existing = client.search_entity(entity_type=models.Person, query={"pref_label": person["pref_label"]}).all()
    if len(existing) > 0:
        print(f"WARNING: Person '{person['pref_label']}' already registered - SKIPPING!")
        continue

    # Create model
    person_model = models.Person(
        pref_label=person["pref_label"],
        family_name=person["family_name"],
        given_name=person["given_name"],
    )

    # Register new entity
    if check_only:
        print(f"CHECK ONLY mode - Would register '{person['pref_label']}': {person_model}")
    else:
        registered_person = client.register_entity(person_model)
        print(f"Successfully registered person: {registered_person}")

        # Save in table
        persons.loc[i, "ID"] = registered_person.id

In [ ]:
persons

In [ ]:
# Write CSV with registered IDs
csv_file = f"registered_persons_{project_context.environment.value}_" + str(datetime.now()).replace(" ", "_").replace(":", "-") + ".csv"
persons.to_csv(os.path.join(output_root, csv_file))

#### Organizations

In [ ]:
for i, organization in organizations.iterrows():
    # Skip if already registered
    existing = client.search_entity(entity_type=models.Organization, query={"pref_label": organization["pref_label"]}).all()
    if len(existing) > 0:
        print(f"WARNING: Organization '{organization['pref_label']}' already registered - SKIPPING!")
        continue

    # Create model
    org_model = models.Organization(
        pref_label=organization["pref_label"],
    )

    # Register new entity
    if check_only:
        print(f"CHECK ONLY mode - Would register '{organization['pref_label']}': {org_model}")
    else:
        registered_org = client.register_entity(org_model)
        print(f"Successfully registered organization: {registered_org}")

        # Save in table
        organizations.loc[i, "ID"] = registered_org.id

In [ ]:
organizations

In [ ]:
# Write CSV with registered IDs
csv_file = f"registered_organizations_{project_context.environment.value}_" + str(datetime.now()).replace(" ", "_").replace(":", "-") + ".csv"
organizations.to_csv(os.path.join(output_root, csv_file))

#### Consortiums

In [ ]:
for i, consortium in consortiums.iterrows():
    # Skip if already registered
    existing = client.search_entity(entity_type=models.Consortium, query={"pref_label": consortium["pref_label"]}).all()
    if len(existing) > 0:
        print(f"WARNING: Consortium '{consortium['pref_label']}' already registered - SKIPPING!")
        continue

    # Create model
    consort_model = models.Consortium(
        pref_label=consortium["pref_label"],
    )

    # Register new entity
    if check_only:
        print(f"CHECK ONLY mode - Would register '{consortium['pref_label']}': {consort_model}")
    else:
        registered_consort = client.register_entity(consort_model)
        print(f"Successfully registered consortium: {registered_consort}")

        # Save in table
        consortiums.loc[i, "ID"] = registered_consort.id

In [ ]:
consortiums

In [ ]:
# Write CSV with registered IDs
csv_file = f"registered_consortia_{project_context.environment.value}_" + str(datetime.now()).replace(" ", "_").replace(":", "-") + ".csv"
consortiums.to_csv(os.path.join(output_root, csv_file))